# Extração de dados

Puxando as 3 fontes e salvando em parquet pra ficar mais rápido de ler depois.
- CSV de transações (financeiro pessoal)
- Ibovespa via yfinance
- Orçamento em Excel

In [ ]:
import os
import pandas as pd
import yfinance as yf

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

## CSV de transações

In [ ]:
# tentei ler normal mas deu erro de encoding, o arquivo deve ter sido salvo no excel em portugues
# df = pd.read_csv("data/source/transacoes_financeiras.csv")
# UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3...

df_csv = pd.read_csv("data/source/transacoes_financeiras.csv", encoding="latin-1")
print(f"{len(df_csv)} registros")
df_csv.head()

In [ ]:
# já dá pra ver que tá bagunçado:
# - coluna 'valor' tem string "R$ ..." misturado com número
# - datas em formato diferente
# - categorias com espaço sobrando
# vou deixar pra limpar no notebook de transformação

df_csv.dtypes

In [ ]:
df_csv.to_parquet("data/raw/transacoes.parquet", index=False)
print("salvo")

## Ibovespa
Usando yfinance pra puxar cotação de 2022 a 2024.

In [ ]:
df_ibov = yf.download("^BVSP", start="2022-01-01", end="2024-12-31", progress=False)
print(f"{len(df_ibov)} pregões")
df_ibov.head()

In [ ]:
# o yfinance retorna MultiIndex nas colunas quando pede 1 ticker só
# descobri isso pq o .to_parquet tava dando erro de serialização
# solução: achatar as colunas

if isinstance(df_ibov.columns, pd.MultiIndex):
    df_ibov.columns = df_ibov.columns.get_level_values(0)
    print("MultiIndex corrigido")

df_ibov = df_ibov.reset_index()
df_ibov.columns

In [ ]:
df_ibov.to_parquet("data/raw/ibovespa.parquet", index=False)
print("salvo")

## Excel de orçamento

In [ ]:
df_xl = pd.read_excel("data/source/orcamento_2022_2024.xlsx", sheet_name="orcamento")
print(f"{len(df_xl)} linhas")
df_xl.head()

In [ ]:
df_xl.to_parquet("data/raw/orcamento.parquet", index=False)
print("salvo")

In [ ]:
# resumo
print(f"transações: {len(df_csv)}")
print(f"ibovespa:   {len(df_ibov)}")
print(f"orçamento:  {len(df_xl)}")
print("\ntudo em data/raw/")